# 8) Convolutional Neural Networks (CNNs)

> NNs for image traitement.

- One of the first complex neural networks to have been considered very useful.

- Nowadays, they have been substituted by LLMs, despite some research still focuses on CNNs.


## THEORY: equivariance for translation and convolutions

-----
THE STARTING PROBLEM: 

We train a multiclassifier NN on the dataset MNIST of images of numbers. We want that it learns to classify the images, thought as sets of pixels, into the corresponding numbers $0,1,\dots,9$.

Ideally, we want that this classification to be invariant by translation, i.e. the same image of a number, if translated, should be classified the same way.

----

**Image**: a matrix $\mathbb{R}^{w\times h}$ of pixels (little squares), or equivalently a function $\phi: [0, h]\times [0, w] \subset \mathbb{R}^2 \rightarrow \mathbb{R}$ such that:

1) $\phi(x)= 0$, $\forall x \notin [0, h]\times [0, w]$, i.e. outside the pixels the image intensity is 0;

2) $\phi(x) = const. \neq 0 $, $\forall x \in [0, h]\times [0, w]$, i.e. on each pixel the image intensity is constant.

Rmk) The second condition could be different, it's arbitrary.

An image $\phi : \mathbb{R}^2 \rightarrow \mathbb{R}$ is on the scale of grays (1 channel == scale of grays).

An image $\phi : \mathbb{R}^2 \rightarrow \mathbb{R}^{3}$ is on the scale of RGB (3 channels == RGB images).

An image $\phi : \mathbb{R}^2 \rightarrow \mathbb{R}^{k}$ is on $k$ different channels.


For the next part of the theory, we assume that images are with compact support (so not necessairly in $[0, h]\times [0, w]$) and smooth, i.e. we assume that
$$
\phi\in C_{c}^{\infty}(\mathbb{R}^{2}).
$$
These assumptions allow us to exploit the functional analysis machinery on images.

------

- **DEFINITION: image invariant by translation:**

Let $\phi: \mathbb{R}^2 \rightarrow \mathbb{R}$ be an image, and consider the two operators corresponding to horizontal and vertical shifts

$$
\tau_{h, \epsilon}, \tau_{w, \epsilon}: \mathbb{R}^{2} \rightarrow \mathbb{R}^{2}
$$

that satisfy

$$
\tau_{h,\epsilon}(x, y) = (x+ \epsilon, y) \quad\text{and}\quad \tau_{w,\epsilon}(x, y) = (x, y+\epsilon).
$$.

The image horizontally translated of $\epsilon$ is $\phi \circ \tau_{h, \epsilon}$. The image vertically translated of $\epsilon$ is $\phi \circ \tau_{w, \epsilon}$.

The image $\phi$ is invariant by translation if $\phi \circ \tau_{h, \epsilon} = \phi \circ \tau_{w, \epsilon} = \phi$, where $\circ$ is the composition.

Rmk) Observe that if an image is invariant by translation, condition 1) can fail ! Indeed, if an image is translated, a pixel outside the borders with intensity zero could be moved within the borders where it should be non-zero.

> Images are NOT invariant by translations, eventually they can approximate it.

-----

*Now, we clarify what transformations == layers we search to work on images.*

- **DEFINITION: transformation on image:**

A transformation over an image $\phi$ is a function
$$
F: C^{\infty}_{c}(\mathbb{R}^{2}) \rightarrow C^{\infty}_{c}(\mathbb{R}^{2}).$$

This class of functions is extremely large. We will consider those functions which satisfy

$$
F_K(\phi)(x,y)
= \int \phi(\xi,\eta)\,K(x,y,\xi,\eta)\,d\xi\,d\eta
$$

for 
$K_i \in C_c^\infty(\mathbb{R}^4)$. 
If the image is $\phi \in C_c^\infty(\mathbb{R}^2)$, then also $F_K(\phi) \in C_c^\infty(\mathbb{R}^2)$ is an image.

Rmk) This restriction is not enough, indeed:
- we consider $\phi$ constant on pixels and every transformation $F$ can we written as above then.
- the transformations $F$ are dense in the set of all transformations, therefore we restrict nothing.


-> *We search for those transformations $F$ which are invariant by translation* i.e. that satisfy
$$
F(\phi \circ \tau_{h, \epsilon}) = F(\phi \circ \tau_{w, \epsilon}) = F(\phi).
$$

Unlukily, no interesting results comes from here. We move towards transformations equivariant for translations.

----

- **DEFINITION: transformation equivariant for translation:**

A transformation $F$, taking in input an image $\phi$, is equivariant for translation if

$$
F(\phi \circ \tau_{h, \epsilon}) = F(\phi) \circ \tau_{h, \epsilon} \quad \text{and} \quad F(\phi \circ \tau_{w, \epsilon}) = F(\phi) \circ \tau_{w, \epsilon},
$$

i.e. a translation of the input produces a corresponding translation of the output / feature map.


>***Inductive bias 1:*** *CNNs are equivariant for translations*.
>
>Each layer of CNNs is (a transformation) equivariant for translations; i.e. if the original image is translated, the output of each layer is translated of the same amount. Also, if the original image is translated, the output is translated of the same amount / its representation in the feature map shifts by the exact same amount.

>VS *invariance for translations*: if the original input image is translated, the image is classified the same. CNNs are NOT invariant for translations.

Rmk) Equivariance preserves how a transformation acts on the output; invariance means that the output does not change at all. CNNs are not invariant by translations, but invariance can be eventually built from equivariance and global average or max pooling.

-------

If we develop the egality above for one of the twos $\tau:=\tau_{\epsilon=1}$ we obtain that the transformation

$$
F(\phi)(x,y) = \iint \phi(\xi,\eta)\Psi(x-\xi,y-\eta)\,d\xi\,d\eta
= (\phi * \Psi)(x,y),
$$

with $K(x,y,\xi,\eta)=\Psi(x-\xi,y-\eta)$, i.e. a *convolution* on $\mathbb{R}^{2}$!

The matrix $K$ is called *kernel* or *filter*.

> ***Inductive bias 2***: *Layers of CNNs are convolutions on $\mathbb{R^{2}}$.*

> Any transformation of an image is expressed by a convolution!

Rmk) Convolutions on images correpond to multiplying the kernel/filter by sovrapposition to the image pixels and sum them.

**Rmk)** *CNNs operations on images are invertible*, because convolutions are invertible functions in the mathematical sense.

## THEORY: the kernel == filter

>The **kernel** is a small matrix containing the weights, that moves across the input image and at each position performs a convolution, i.e.
>1) multiplies the corresponding image pixels by sovrapposition, 
>2) sums the results.

> **Kernel size**: the height and width of the filter used by convolution.

> **Number of parameters / weights to learn**: the dimension of the kernel.

Often, kernels are small squared matrices of dimension 'kernel_size $\times$ 
kernel size'. Therefore, the number of parameters to learn is kernel_size^2.

Rmk) In PyTorch, kernels are initialised randomly by default.

> ***Inductive bias 3***: *parameters / weights sharing*, because the CNN parameters belong to the kernel.

> ***Inductive bias 4***: *local connectivity*, i.e. the kernel convolution reads each pixel "locally" considering its nearby neightbors to whom it could be related.

In [6]:
import torch

In [ ]:
# A 2dim CONVOLUTIONAL LAYER, on scale of grays

layer = torch.nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3)

# 9 parameters to learn

In [10]:
print(layer.weight.shape)
print(layer.weight)
print(layer.bias)

torch.Size([1, 1, 3, 3])
Parameter containing:
tensor([[[[-0.0644, -0.0148, -0.3215],
          [-0.0809, -0.0064, -0.0227],
          [-0.2436,  0.2544, -0.2023]]]], requires_grad=True)
Parameter containing:
tensor([-0.3107], requires_grad=True)


### Gaussian kernel

In [12]:
diag = torch.tensor([[1.0, 0, 0],
                    [0, 1.0, 0],
                    [0, 0, 1.0]],
                    requires_grad=False).view(1,1,3,3)   # unsqueeze
bias = -2

x
=
torch
.
tensor
(
images
[
0
]
.
reshape
(
1
,
1
,
28
,
28
))
/
255.0

with
torch
.
no_grad
():
couche
.
weight
.
copy_
(
diag
)

couche
.
bias
.
copy_
(
bias
)
y
=
couche
(
x
)

def
plot
(
x
,
y
,
figsize
=
(
6
,
3
),
cmap
=
None
):
_
,
axes
=
plt
.
subplots
(
1
,
2
,
figsize
=
figsize
)
axes
[
0
]
.
imshow
(
x
.
squeeze
(),
cmap
=
cmap
);
axes
[
0
]
.
set_title
(
'Image originelle'
)
axes
[
1
]
.
imshow
(
np
.
maximum
(
y
.
squeeze
(),
0
))
axes
[
1
]
.
set_title
(
'Image transformée'
)
plot
(
x
,
y
)



SyntaxError: invalid syntax (243789552.py, line 8)

## THEORY: padding

When the filter reads an input image and applies convolution, the feature map / output is shrinked.

--> **Padding**: adds zeros all around the borders of the image, to avoid output to be shriked.

For example, with padding=1 we add all around the image one single border of zeros.

## ImageNet:

**ImageNet**: *the most known dataset of labelled images worldwide, to evaluate new models.* It contains 14 million images, labelled into 1000 classes.

-> Enourmous dataset, which we will not be able to download due to its size.

-> Requires days of time for a single evaluation, even with GPUs.

-> It is common to extract with 'Subsets' only the desidered part, and work on this smaller dataset.

## ResNet:

*Without skip connections:* the loss function is generally more complex with many peaks and local minima and flat regions. With 20 arrows/layers in the NN, the surface is already enough complicated.

*With skip connections:* the loss function is generally smooth and eventually with a unique global minimum. New layers connections are defined by $y = f(x) + x$ and called **residual connections**.

> **Skip connections** improves the surface of the loss function and thus, optimizing the weights.

> The idea is "externally" connecting non-successive layers and skipping the within internal successive connections. This lowers the number of parameters to learn.

In [ ]:
# BASIC BLOCK

class BasicBlock(nn.Module):
    def __init__(self, inplanes, planes, stride):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, bias=False, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLu(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self.make_resnet_layer(BasicBlock, 64, 64, 3)
        self.layer2 = self.make_resnet_layer(BasicBlock, 64, 128, 4, stride=2)
        self.layer3 = self.make_resnet_layer(BasicBlock, 128, 256, 6, stride=2)
        self.layer4 = self.make_resnet_layer(BasicBlock, 256, 512, 3, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = 

In [ ]:
resnet34 = ResNet(num_classes = 10)

## What are the new elements wrto VGG ?

- residual connections (skip connections): connects non successive layers and skip the connections within.
- global pooling: contract all the pixels into a single one (?).
- batch normalization: normalize the batches.
- bottleneck: 
- dropout: while weights are optimised, deactivate at each run some neurons; to avoid that that the NN relies to much only on specific neurons.